[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimyortega55-collab/motor-OCR/blob/main/entrenamiento/colab_extraer_muestra_evaluacion.ipynb)

# Extraer muestra de evaluacion (manifiesto.jsonl) en Colab

Corre `entrenamiento/extraer_muestra_evaluacion.py` sobre los 11 PDF de
`pruebas/pdfs_de_prueba/` con GPU en vez de en CPU local. El script corre
Capas 1-2 (triage + segmentacion) y llama a `ocr_formula` (pix2tex) una vez
por cada fragmento de formula que Capa 3 recortaria en produccion -no por
bloque/parrafo completo-, asi que el cuello de botella es la cantidad de
formulas del corpus, no el tamano de los PDF.

**Antes de correr esto necesitas subir los PDF a tu Drive una sola vez**:
`pruebas/` esta en `.gitignore` (material interno, no se publica en
GitHub -ver `POLITICA_PUBLICACION.md`-), asi que el clon del repo no los trae.
Sube la carpeta `pruebas/pdfs_de_prueba/` completa (los 11 `.pdf`) a
`MyDrive/motor-ocr-finetuning/pdfs_de_prueba/` antes de correr la celda 4.

## 1. Confirmar que hay GPU asignada

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No hay GPU asignada. Entorno de ejecucion -> Cambiar tipo de entorno de "
    "ejecucion -> GPU, y volve a correr desde esta celda."
)
gpu = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | {gpu.name} | {gpu.total_memory / 1024**3:.1f} GB")

## 2. Clonar el repo

Clona `origin/main`: si tocaste el script en local, pushealo antes de abrir
este notebook.

In [ ]:
import os

REPO_URL = "https://github.com/rimyortega55-collab/motor-OCR.git"
REPO_DIR = "/content/motor-OCR"

if os.path.isdir(REPO_DIR):
    !git -C /content/motor-OCR pull --ff-only
else:
    !git clone {REPO_URL} /content/motor-OCR

%cd /content/motor-OCR
!git log --oneline -1

## 3. Instalar dependencias

El script importa `motor_ocr.layout` y `motor_ocr.triage`, que a su vez
importan el resto del paquete (easyocr, doctr, pix2tex, transformers...),
asi que -a diferencia de la prueba de humo de fine-tuning, que solo necesita
pix2tex- aca hace falta instalar el paquete completo.

In [ ]:
!pip install -q -e .

import torch

# La instalacion puede arrastrar una version de torch distinta a la que Colab
# trae preinstalada y dejar la GPU sin usar.
print("torch:", torch.__version__, "| cuda disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Se perdio el acceso a la GPU al instalar dependencias. Entorno de ejecucion "
    "-> Reiniciar entorno de ejecucion, y volve a correr desde la celda del clon "
    "(no hace falta reinstalar: los paquetes sobreviven al reinicio)."
)

## 4. Traer los PDF de prueba desde Drive

`pruebas/` no viene en el clon (ver nota de arriba). Se copian desde Drive a
la ruta que el script espera.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/motor-ocr-finetuning")

origen_pdfs = DRIVE_DIR / "pdfs_de_prueba"
destino_pdfs = Path("pruebas/pdfs_de_prueba")
destino_pdfs.mkdir(parents=True, exist_ok=True)

assert origen_pdfs.is_dir(), (
    f"No encontre {origen_pdfs}. Subi la carpeta pruebas/pdfs_de_prueba/ "
    "completa (los 11 .pdf) a esa ruta de tu Drive antes de correr esta celda."
)

pdfs = sorted(origen_pdfs.glob("*.pdf"))
assert pdfs, f"{origen_pdfs} esta vacia -no hay ningun .pdf para copiar."

for pdf in pdfs:
    destino = destino_pdfs / pdf.name
    if not destino.exists():
        destino.write_bytes(pdf.read_bytes())

print(f"{len(pdfs)} PDF copiados a {destino_pdfs}/")

## 5. Correr la extraccion

Reescribe `entrenamiento/evaluacion_real/manifiesto.jsonl` tras cada PDF (no
solo al final), asi que si Colab desconecta a mitad de camino no se pierde lo
ya procesado -volves a correr esta celda y sigue de donde quedo, sin repetir
los PDF que ya tienen entradas en el manifiesto.

In [ ]:
!python entrenamiento/extraer_muestra_evaluacion.py

## 6. Guardar el resultado en Drive

El disco de la VM se borra al desconectarse. `entrenamiento/evaluacion_real/`
(manifiesto + recortes PNG) queda en Drive, mismo lugar que los demas
artefactos del proyecto.

In [ ]:
import shutil

destino = DRIVE_DIR / "evaluacion_real"
shutil.copytree("entrenamiento/evaluacion_real", destino, dirs_exist_ok=True)

import json

with open("entrenamiento/evaluacion_real/manifiesto.jsonl", encoding="utf-8") as fh:
    filas = [json.loads(linea) for linea in fh]

print(f"{len(filas)} recortes -> {destino}")
print("Por PDF:", {pdf: sum(1 for f in filas if f["pdf"] == pdf) for pdf in sorted({f["pdf"] for f in filas})})